# 📄 RAG Document Q&A — Local Pipeline with Ollama & FAISS

**Author:** Davide Barbato  
**Stack:** LangChain · Ollama · FAISS · nomic-embed-text · Llama 3.2

---

## 🎯 Project Goal

Build a fully **local** Retrieval-Augmented Generation (RAG) pipeline that allows users to:
1. Upload any PDF document
2. Ask questions in natural language
3. Get accurate answers grounded in the document content

Everything runs **on-device** — no API keys, no data sent to the cloud.

---

## 🏗️ Architecture Overview

```
┌─────────────┐    ┌──────────────────┐    ┌───────────────┐
│   PDF File  │───▶│  Text Splitting  │───▶│  Embeddings   │
│             │    │  (chunks ~500    │    │  nomic-embed  │
│             │    │   tokens each)   │    │  -text        │
└─────────────┘    └──────────────────┘    └───────┬───────┘
                                                   │
                                                   ▼
┌─────────────┐    ┌──────────────────┐    ┌───────────────┐
│   Answer    │◀───│   Llama 3.2:3b   │◀───│  FAISS Vector │
│  + Sources  │    │   (generation)   │    │     Store     │
└─────────────┘    └──────────────────┘    └───────────────┘
                            ▲
                   ┌────────┴────────┐
                   │  User Question  │
                   └─────────────────┘
```

**Key concepts:**
- **Chunking**: Split the document into small overlapping pieces so each chunk fits in the LLM context window
- **Embeddings**: Convert text chunks into numerical vectors that capture semantic meaning
- **Vector Store (FAISS)**: Index that enables fast similarity search over all chunk vectors
- **Retrieval**: Given a question, find the top-k most semantically similar chunks
- **Generation**: Feed retrieved chunks as context to the LLM to generate a grounded answer

---

## ⚙️ Requirements

Before running this notebook, make sure you have:
- [Ollama](https://ollama.com) installed and running
- Models pulled: `ollama pull llama3.2:3b` and `ollama pull nomic-embed-text`
- Dependencies installed: `pip install -r requirements.txt`

---
## 📦 Step 1 — Imports & Configuration

We import all required libraries and define the model names and prompt template.

- `PyPDFLoader` — reads and parses PDF files page by page
- `RecursiveCharacterTextSplitter` — splits text into chunks respecting natural boundaries (paragraphs, sentences)
- `OllamaEmbeddings` — generates vector embeddings using the local `nomic-embed-text` model
- `ChatOllama` — interface to the local `llama3.2:3b` LLM
- `FAISS` — fast in-memory vector similarity search library by Meta
- `RetrievalQA` — LangChain chain that combines retrieval + generation

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL   = "llama3.2:3b"

PDF_PATH = "/Users/davidebarbato/Downloads/Master Thesis.pdf"

print("✅ Libraries imported successfully")
print(f"   LLM Model     : {LLM_MODEL}")
print(f"   Embed Model   : {EMBED_MODEL}")

---
## 📝 Step 2 — Load & Split the PDF

### Why do we split the document into chunks?

LLMs have a limited **context window** — they can only process a fixed amount of text at once.
Instead of feeding the entire document, we split it into small overlapping chunks and only retrieve
the most relevant ones for each question.

### Key parameters:
| Parameter | Value | Description |
|-----------|-------|-------------|
| `chunk_size` | 500 | Max number of characters per chunk |
| `chunk_overlap` | 50 | Characters shared between adjacent chunks (preserves context at boundaries) |
| `separators` | `["\n\n", "\n", ".", " "]` | Tries to split at paragraph → sentence → word level |

### Why overlap?
Without overlap, a sentence split across two chunks would lose context.
With overlap, both chunks contain part of that sentence — retrieval is more robust.

In [ ]:
def load_and_split(pdf_path: str, chunk_size: int = 500, chunk_overlap: int = 50):
    """
    Load a PDF document and split it into overlapping text chunks.
    
    Args:
        pdf_path     : Path to the PDF file
        chunk_size   : Maximum characters per chunk
        chunk_overlap: Characters shared between adjacent chunks
    
    Returns:
        List of Document objects (chunk text + metadata)
    """
    # Load the PDF — each page becomes a separate Document
    loader    = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"📄 Loaded {len(documents)} pages from '{pdf_path}'")

    # Split documents into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "]
    )
    chunks = splitter.split_documents(documents)
    print(f"✂️  Created {len(chunks)} chunks  "
          f"(size={chunk_size}, overlap={chunk_overlap})")
    return chunks


# ── Run ───────────────────────────────────────────────────────
chunks = load_and_split(PDF_PATH)

# Inspect a sample chunk
print("\n--- Sample chunk ---")
print(f"Content  : {chunks[0].page_content[:200]}...")
print(f"Metadata : {chunks[0].metadata}")

---
## 🔢 Step 3 — Generate Embeddings & Build Vector Store

### What are embeddings?

An **embedding** is a dense numerical vector that captures the semantic meaning of a text.
Texts with similar meaning will have vectors that are close together in high-dimensional space.

We use `nomic-embed-text`, a lightweight open-source model that generates **768-dimensional vectors**.

### What is FAISS?

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search.
It indexes all chunk vectors so we can instantly find the top-k most similar chunks to a query.

**Similarity metric used:** cosine similarity (how aligned two vectors are in direction)

```
Query: "What is the main conclusion?"  →  embed  →  [0.12, -0.34, 0.87, ...]
                                                              ↓ cosine similarity
Chunk 1: "The study concludes that..."  →  embed  →  [0.11, -0.31, 0.90, ...]  ← HIGH similarity
Chunk 2: "The dataset contains 500..."  →  embed  →  [-0.5, 0.22, -0.10, ...]  ← LOW similarity
```

In [ ]:
def build_vectorstore(chunks):
    """
    Generate embeddings for all chunks and index them in a FAISS vector store.
    
    Args:
        chunks: List of Document objects from load_and_split()
    
    Returns:
        FAISS vector store ready for similarity search
    """
    print(f"🔢 Generating embeddings with '{EMBED_MODEL}'...")
    print(f"   This may take a moment for large documents...")

    embeddings  = OllamaEmbeddings(model=EMBED_MODEL)
    vectorstore = FAISS.from_documents(chunks, embeddings)

    print(f"✅ Vector store built — {len(chunks)} vectors indexed")
    return vectorstore


# ── Run ───────────────────────────────────────────────────────
vectorstore = build_vectorstore(chunks)

# Test similarity search directly
print("\n--- Similarity search test ---")
test_query   = "What is the main topic of this document?"
top_docs     = vectorstore.similarity_search(test_query, k=2)
print(f"Query  : '{test_query}'")
print(f"Top-1  : {top_docs[0].page_content[:150]}...")

---
## 🔗 Step 4 — Build the RAG Chain

### How does the RAG chain work?

The `RetrievalQA` chain combines two components:

1. **Retriever** — given a question, searches FAISS for the top-k most relevant chunks
2. **Generator** — feeds the question + retrieved chunks into the LLM to produce a grounded answer

### Why a custom prompt?

The default LangChain prompt is generic. Our custom prompt:
- Explicitly tells the LLM to use **only** the provided context
- Instructs it to admit when the answer is not in the document
- Reduces hallucinations significantly

### Temperature = 0.1

A low temperature makes the LLM more **deterministic and factual** — ideal for Q&A tasks
where we want precise answers, not creative ones.

In [ ]:
# ── Custom prompt template ────────────────────────────────────
PROMPT_TEMPLATE = """You are an expert document analysis assistant.
Use ONLY the provided context to answer the question.
If the answer is not in the context, clearly state that you don't know.

Context:
{context}

Question: {question}

Detailed answer:"""


def build_rag_chain(vectorstore, k: int = 4):
    """
    Build the full RAG retrieval-augmented generation chain.
    
    Args:
        vectorstore: FAISS vector store with indexed chunks
        k          : Number of chunks to retrieve per query
    
    Returns:
        RetrievalQA chain ready for querying
    """
    # Initialize the local LLM
    llm = ChatOllama(
        model=LLM_MODEL,
        temperature=0.1,   # low = more deterministic and factual
    )

    # Build the prompt
    prompt = PromptTemplate(
        template=PROMPT_TEMPLATE,
        input_variables=["context", "question"]
    )

    # Combine retriever + LLM into a single chain
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",          # 'stuff' = concatenate all chunks into one prompt
        retriever=vectorstore.as_retriever(search_kwargs={"k": k}),
        return_source_documents=True, # return which chunks were used
        chain_type_kwargs={"prompt": prompt}
    )

    print(f"✅ RAG chain built")
    print(f"   LLM        : {LLM_MODEL} (temp={0.1})")
    print(f"   Retrieval  : top-{k} chunks per query")
    print(f"   Chain type : stuff (all chunks concatenated)")
    return chain


# ── Run ───────────────────────────────────────────────────────
chain = build_rag_chain(vectorstore, k=4)

---
## 💬 Step 5 — Ask Questions

Now we can query the document in natural language.

For each question the pipeline:
1. Embeds the question into a vector
2. Searches FAISS for the top-4 most similar chunks
3. Feeds the question + chunks to Llama 3.2
4. Returns the answer and the source chunks used

In [ ]:
def ask(chain, question: str, show_sources: bool = True):
    """
    Ask a question and display the answer with its source chunks.
    
    Args:
        chain       : RAG chain from build_rag_chain()
        question    : Natural language question
        show_sources: Whether to print the retrieved source chunks
    
    Returns:
        answer (str), sources (list of Documents)
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")

    result  = chain.invoke({"query": question})
    answer  = result["result"]
    sources = result["source_documents"]

    print(f"\n💡 Answer:\n{answer}")

    if show_sources:
        print(f"\n📚 Sources used ({len(sources)} chunks):")
        for i, doc in enumerate(sources, 1):
            page = doc.metadata.get('page', 'N/A')
            print(f"  [{i}] Page {page + 1}: {doc.page_content[:120]}...")

    return answer, sources

In [ ]:
answer, sources = ask(chain, "What is this thesis about? What is the main research question?")

In [ ]:
answer, sources = ask(chain, "Which NLP or machine learning models are used in this thesis?")

In [ ]:
answer, sources = ask(chain, "Which social media platforms were analyzed in this thesis and why were they chosen?")

---
## 🔁 Step 6 — Interactive Q&A Loop

Run this cell for an interactive session — type your questions directly and get answers in real time.
Type `exit` to stop.

In [ ]:
import sys

def interactive_qa(chain):
    """Interactive Q&A loop — skipped when running non-interactively (e.g. nbconvert)."""
    print("Interactive Q&A — type 'exit' to quit\n")
    while True:
        try:
            question = input("Your question: ").strip()
        except Exception:
            print("(non-interactive environment — skipping input loop)")
            break
        if question.lower() in ["exit", "quit", "q"]:
            print("Session ended.")
            break
        if question:
            ask(chain, question, show_sources=False)

interactive_qa(chain)

---
## 📊 Step 7 — Pipeline Evaluation

A good portfolio project always includes some form of evaluation.
Here we test the pipeline on a small set of question-answer pairs and measure basic metrics.

In [ ]:
questions_10 = [
    "What is the time period covered by the data collected in this thesis?",
    "Which geopolitical events are analyzed in this thesis?",
    "How are the geopolitical events categorized or classified?",
    "What are the main findings and conclusions of the thesis?",
    "What datasets were used and how was the data collected?",
    "How does sentiment from each social media platform differ in predicting financial markets?",
    "What are the original contributions of this thesis to the existing literature?",
]

print("Running 10-question evaluation on the Master Thesis...\n")
results = []

for question in questions_10:
    answer, sources = ask(chain, question, show_sources=False)
    results.append({
        "question"     : question,
        "answer"       : answer,
        "num_sources"  : len(sources),
        "answer_length": len(answer.split()),
    })

print("\n" + "="*70)
print("EVALUATION SUMMARY — Master Thesis Q&A")
print("="*70)
print(f"{'#':<4} {'Question':<45} {'Words':>6} {'Sources':>8}")
print("-"*70)
for i, r in enumerate(results, 1):
    q = r['question'][:43] + ".." if len(r['question']) > 43 else r['question']
    print(f"{i:<4} {q:<45} {r['answer_length']:>6} {r['num_sources']:>8}")
print("-"*70)
avg_words   = sum(r['answer_length'] for r in results) / len(results)
avg_sources = sum(r['num_sources']   for r in results) / len(results)
print(f"{'Average':<49} {avg_words:>6.1f} {avg_sources:>8.1f}")

---
## 🧠 Step 8 — How to Save & Reload the Vector Store

Building embeddings takes time. FAISS lets us **save the index to disk** so we don't need to rebuild it every time.

In [ ]:
import os

VECTORSTORE_PATH = "faiss_index"

# ── Save ──────────────────────────────────────────────────────
vectorstore.save_local(VECTORSTORE_PATH)
print(f"✅ Vector store saved to '{VECTORSTORE_PATH}/'")

# ── Reload ────────────────────────────────────────────────────
embeddings          = OllamaEmbeddings(model=EMBED_MODEL)
vectorstore_reloaded = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True   # required by FAISS
)
print(f"✅ Vector store reloaded from '{VECTORSTORE_PATH}/'")

# Rebuild chain from reloaded store
chain_reloaded = build_rag_chain(vectorstore_reloaded, k=4)
print("✅ Chain rebuilt from saved index — ready to use")

---
## ✅ Summary & Next Steps

### What we built

| Component | Tool | Purpose |
|-----------|------|---------|
| Document Loader | `PyPDFLoader` | Parse PDF pages |
| Text Splitter | `RecursiveCharacterTextSplitter` | Split into overlapping chunks |
| Embeddings | `nomic-embed-text` via Ollama | Vectorize text semantically |
| Vector Store | `FAISS` | Fast similarity search |
| LLM | `llama3.2:3b` via Ollama | Generate grounded answers |
| Chain | `RetrievalQA` | Connect all components |

### 🚀 Possible Extensions

- **Multi-document support** — index multiple PDFs into the same vector store
- **Conversation memory** — add `ConversationBufferMemory` for multi-turn Q&A
- **Hybrid search** — combine semantic search (FAISS) with keyword search (BM25)
- **Streamlit UI** — wrap this pipeline into a web app (see `app.py`)
- **Evaluation with RAGAS** — measure faithfulness, answer relevance, context recall
- **Swap LLM** — replace `llama3.2:3b` with a larger model for better quality

---

### 📖 References

- [LangChain Documentation](https://docs.langchain.com)
- [Ollama Model Library](https://ollama.com/library)
- [FAISS GitHub](https://github.com/facebookresearch/faiss)
- [RAG Paper — Lewis et al. 2020](https://arxiv.org/abs/2005.11401)
- [nomic-embed-text](https://ollama.com/library/nomic-embed-text)